# 검색 베이스라인 (holdout test 350 + 변형 1,400)

`data/derived/medical_test_350_expanded.parquet`(원본 350 × 변형 4종)로 **검색 단계만** 다시 채점합니다. LLM 생성은 하지 않습니다.

**흐름**

```
질문 ─┬─ bge 임베딩 → FAISS 상위 100 ─────────┬─ (Q-dense)
      │                                       ├─ RRF 결합 → (Q-hybrid)
      └─ BM25 단어 검색 → 상위 100 ───────────┘
Q-dense 후보 100 → cross-encoder 재정렬 → 최종 순위 → 상위 20을 생성 단계로
```

| 방법 | 설명 |
|---|---|
| Q-bm25 | BM25 단어 검색 |
| Q-dense | `bge-base-en-v1.5` 임베딩 + FAISS `IndexFlatIP` |
| Q-hybrid | 위 두 결과를 RRF로 합침 |
| Q-dense → MiniLM-L-6 | dense 후보 100을 `ms-marco-MiniLM-L-6-v2`로 재정렬 (04 챗봇에서 쓰는 설정) |
| Q-dense → bge-reranker-base | dense 후보 100을 `bge-reranker-base`로 재정렬 |

**DB**: `T1`/`T2 (v1.1)`이 만든 `rag_msmarco/v11_test`(문단 142,605개)를 그대로 씁니다. 테스트 질문은 MS MARCO **v1.1 train**에서 뽑았고, 이 DB에는 v1.1 train 건강 질문의 문단이 모두 들어 있습니다. DB에 없는 정답 문단이 있으면 메모리에서만 추가하고 개수를 출력합니다. 드라이브의 인덱스 파일은 바꾸지 않습니다.

**정답 문단**: MS MARCO v1.1 train에서 같은 `query_id`의 `is_selected=1` 문단 전부 + parquet의 `context`.

**주의할 점**
- **학습 데이터 누출 가능성**: `ms-marco-MiniLM-L-6-v2`는 MS MARCO train 질문으로 학습된 모델이고, 테스트 질문도 MS MARCO train에서 나왔습니다. 따라서 재정렬 점수가 실제보다 높게 나올 수 있습니다. `bge` 계열도 학습 데이터에 MS MARCO가 들어 있습니다.
- **`핵심 단어 회피` 변형**은 주제어를 지운 질문입니다(예: "From the passage, what does this medicine do?"). 지문이 함께 주어진다는 가정이라서, 검색 점수는 거의 0이 나오는 게 정상입니다. 검색 성능은 `원본`·`구어체`·`쉬운 말`로 보세요.
- 이 노트북에는 방법을 고르는 단계가 없습니다(고정 설정으로 한 번만 채점). 이 결과를 보고 설정을 바꾸면 350개는 더 이상 독립된 시험 세트가 아닙니다.

**런타임: A100(T4도 가능, 재정렬이 느림).** 위에서부터 순서대로 실행하세요.

## 1. 설치 + 저장 위치

In [13]:
!pip -q install sentence-transformers faiss-cpu bm25s

In [14]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as e:
    raise RuntimeError('드라이브 연결이 필요합니다. DB와 테스트 파일이 드라이브에 있어요.') from e

DB_DIR    = '/content/drive/MyDrive/rag_msmarco/v11_test'                               # T1·T2 (v1.1) 결과
TEST_PATH = '/content/drive/MyDrive/RAG-Chat/data/medical_test_350_expanded.parquet'   # 변형 포함 1,400행
OUT_DIR   = '/content/drive/MyDrive/RAG-Chat/results/baseline_retrieval'
print('DB:', DB_DIR, '\n테스트:', TEST_PATH, '\n결과 저장:', OUT_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
DB: /content/drive/MyDrive/rag_msmarco/v11_test 
테스트: /content/drive/MyDrive/RAG-Chat/data/medical_test_350_expanded.parquet 
결과 저장: /content/drive/MyDrive/RAG-Chat/results/baseline_retrieval


## 2. 설정

`LIMIT`에 숫자를 넣으면 원본 질문을 그만큼만 써서 빠르게 시험합니다(변형도 같이 줄어듭니다). 본 실행은 `None`.

In [15]:
import os, re, json, time
from collections import Counter
import numpy as np
import pandas as pd
import torch

K         = 100        # 1단계 후보 수 (= 재정렬할 후보 수)
RRF_K     = 60         # RRF 결합 상수 (관례값)
TOP_SAVE  = 20         # 생성 단계로 넘길 문단 수
LIMIT     = None       # 빠른 시험용. 예: 20

EMB_MODEL    = 'BAAI/bge-base-en-v1.5'
QUERY_PREFIX = 'Represent this sentence for searching relevant passages: '
RERANKERS    = ['cross-encoder/ms-marco-MiniLM-L-6-v2', 'BAAI/bge-reranker-base']
FINAL        = 'Q-dense → ms-marco-MiniLM-L-6-v2'      # 생성 단계로 넘길 방법 (04 챗봇과 같은 설정)
VARIANTS     = ['원본', '구어체', '쉬운 말', '핵심 단어 회피']

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
os.makedirs(OUT_DIR, exist_ok=True)
print('장치:', DEVICE)

def load_jsonl(path):
    with open(path, encoding='utf-8') as f:
        return [json.loads(line) for line in f]

def save_jsonl(rows, path):
    with open(path, 'w', encoding='utf-8') as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + '\n')

def clean_query(q):
    q = ' '.join(q.split())
    return re.sub(r'^[\s.,;:!?\-]+', '', q)

def clean_text(t):
    return ' '.join(t.split())

장치: cuda


## 3. DB 불러오기 (없으면 새로 만들기)

`DB_DIR`에 `corpus.jsonl`과 `faiss_bge_flatip.index`가 없으면 `T1`·`T2 (v1.1)`과 **같은 방식, 같은 시드**로 DB를 새로 만들어 저장합니다(A100 기준 약 5~10분).
- 문단 = v1.1 dev 건강 질문 문단 + train 건강 질문 문단 전부 + dev 일반 질문 3,000개의 문단 (중복 제거)
- 순서와 시드가 같으므로 pid도 T1과 같게 나옵니다.

다른 위치에 이미 만든 DB가 있으면 1번 셀의 `DB_DIR`만 바꾸면 됩니다.

In [16]:
import random, faiss
from urllib.parse import urlparse
from datasets import load_dataset

HEALTH_SITES = [
    'webmd.com', 'mayoclinic.org', 'medicinenet.com', 'medical-dictionary.thefreedictionary.com',
    'verywell.com', 'healthline.com', 'drugs.com', 'livestrong.com', 'healthtap.com', 'cdc.gov',
    'rightdiagnosis.com', 'emedicinehealth.com', 'medscape.com', 'everydayhealth.com',
    'medlineplus.gov', 'nhs.uk', 'rxlist.com', 'sharecare.com', 'medicalnewstoday.com',
    'kidshealth.org', 'nih.gov', 'patient.info', 'labtestsonline.org', 'aafp.org',
    'clevelandclinic.org', 'hopkinsmedicine.org', 'healthgrades.com', 'familydoctor.org',
    'merckmanuals.com', 'uptodate.com', 'health.harvard.edu', 'babycenter.com', 'psychologytoday.com',
]
MIN_HEALTH_HITS, N_DISTRACTOR_Q, DB_SEED = 2, 3000, 42      # T1과 같은 값

def is_health(url):
    h = urlparse(url).netloc.lower()
    h = h[4:] if h.startswith('www.') else h
    return any(h == s or h.endswith('.' + s) for s in HEALTH_SITES)

def count_hits(split):
    hits = []
    for b in split.select_columns(['passages']).iter(batch_size=5000):
        hits += [sum(is_health(u) for u in p['url']) for p in b['passages']]
    return np.array(hits)

def iter_rows(split, idx):
    for b in split.select(list(map(int, idx))).iter(batch_size=1000):
        for j in range(len(b['query'])):
            yield {k: b[k][j] for k in b}

def build_v11_db():
    """T1(문단 모음) + T2(임베딩·FAISS)를 한 번에 다시 만듭니다."""
    from sentence_transformers import SentenceTransformer
    ds = load_dataset('microsoft/ms_marco', 'v1.1')
    dev, train = ds['validation'], ds['train']
    dev_hits, train_hits = count_hits(dev), count_hits(train)

    rng = random.Random(DB_SEED)
    dev_other_idx = np.where(dev_hits == 0)[0].tolist()
    rng.shuffle(dev_other_idx)
    sources = [(dev,   np.where(dev_hits >= MIN_HEALTH_HITS)[0],   'dev_health'),
               (train, np.where(train_hits >= MIN_HEALTH_HITS)[0], 'train_health'),
               (dev,   sorted(dev_other_idx[:N_DISTRACTOR_Q]),     'dev_other')]
    corpus, seen = [], set()
    for split, idx, src in sources:
        for r in iter_rows(split, idx):
            for text, url in zip(r['passages']['passage_text'], r['passages']['url']):
                text = clean_text(text)
                if text and text not in seen:
                    seen.add(text)
                    corpus.append({'pid': len(corpus), 'text': text, 'url': url, 'src': src})
    os.makedirs(DB_DIR, exist_ok=True)
    save_jsonl(corpus, f'{DB_DIR}/corpus.jsonl')
    print(f'문단 {len(corpus):,}개', Counter(c['src'] for c in corpus))

    model = SentenceTransformer(EMB_MODEL, device=DEVICE)
    if DEVICE == 'cuda':
        model.half()
    model.max_seq_length = 256
    emb = model.encode([c['text'] for c in corpus], batch_size=512, normalize_embeddings=True,
                       convert_to_numpy=True, show_progress_bar=True).astype('float16')   # T2와 같게 float16 저장
    np.save(f'{DB_DIR}/corpus_emb_bge.npy', emb)
    index = faiss.IndexFlatIP(emb.shape[1])
    index.add(emb.astype('float32'))
    faiss.write_index(index, f'{DB_DIR}/faiss_bge_flatip.index')
    del model

if not all(os.path.exists(f'{DB_DIR}/{n}') for n in ['corpus.jsonl', 'faiss_bge_flatip.index']):
    print('DB가 없어서 새로 만듭니다 →', DB_DIR)
    t = time.time()
    build_v11_db()
    print(f'DB 생성 {time.time() - t:.0f}초')

DB가 없어서 새로 만듭니다 → /content/drive/MyDrive/rag_msmarco/v11_test


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


README.md:   0%|          | 0.00/9.48k [00:00<?, ?B/s]

v1.1/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 21.4MB            

v1.1/validation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

v1.1/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  175MB            

... (18 outputs left out)

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/279 [00:00<?, ?it/s]

DB 생성 390초


In [17]:
corpus = load_jsonl(f'{DB_DIR}/corpus.jsonl')
index  = faiss.read_index(f'{DB_DIR}/faiss_bge_flatip.index')
texts  = [c['text'] for c in corpus]
assert index.ntotal == len(corpus), '인덱스와 문단 수가 다릅니다'
print(f'문단 {len(corpus):,} / 인덱스 {index.ntotal:,}')

문단 142,605 / 인덱스 142,605


## 4. 테스트 질문 + 정답 문단 번호

- MS MARCO v1.1 train에서 같은 `query_id`의 정답 문단(`is_selected=1`)을 모두 가져옵니다. parquet의 `context`도 정답에 넣습니다.
- 정답 문단 내용을 DB 문단과 똑같이 정리(`clean_text`)한 뒤 맞춰서 pid를 찾습니다.
- DB에 없는 정답 문단은 `texts` 끝에 붙입니다. FAISS에는 다음 셀에서 메모리로만 추가합니다.

In [18]:
from datasets import load_dataset

df = pd.read_parquet(TEST_PATH)
assert set(df['query_variant_type']) == set(VARIANTS), df['query_variant_type'].unique()
if LIMIT:
    keep = df['comparison_group_id'].drop_duplicates().head(LIMIT)
    df = df[df['comparison_group_id'].isin(keep)].reset_index(drop=True)
need = {int(q) for q in df['source_query_id']}

selected = {}
train = load_dataset('microsoft/ms_marco', 'v1.1')['train']
for b in train.select_columns(['query_id', 'passages']).iter(batch_size=5000):
    for qid, p in zip(b['query_id'], b['passages']):
        if int(qid) in need:
            selected[int(qid)] = [clean_text(t) for t, s in zip(p['passage_text'], p['is_selected']) if s == 1]
print(f'MS MARCO에서 찾은 원본 질문 {len(selected)} / {len(need)}')

pid_of, n_db = {t: i for i, t in enumerate(texts)}, len(texts)
qs = []
for r in df.itertuples():
    gold_texts = dict.fromkeys(selected.get(int(r.source_query_id), []) + [clean_text(r.context)])
    gold = []
    for t in gold_texts:
        if t not in pid_of:                       # DB에 없는 정답 문단 → 끝에 추가
            pid_of[t] = len(texts)
            texts.append(t)
        gold.append(pid_of[t])
    qs.append({'qid': int(r.query_id), 'group': int(r.comparison_group_id), 'variant': r.query_variant_type,
               'category': r.question_category, 'query': clean_query(r.query),
               'answers': [str(a) for a in r.answers], 'gold_pids': sorted(set(gold))})

added = texts[n_db:]
in_db = sum(any(p < n_db for p in q['gold_pids']) for q in qs if q['variant'] == '원본')
n_orig = sum(q['variant'] == '원본' for q in qs)
print(f'질문 {len(qs)}개 {dict(Counter(q["variant"] for q in qs))}')
print(f'정답 문단이 원래 DB에 있던 원본 질문: {in_db} / {n_orig}  (메모리에 추가한 문단 {len(added)}개)')

MS MARCO에서 찾은 원본 질문 350 / 350
질문 1400개 {'원본': 350, '구어체': 350, '쉬운 말': 350, '핵심 단어 회피': 350}
정답 문단이 원래 DB에 있던 원본 질문: 308 / 350  (메모리에 추가한 문단 48개)


## 5. 공통 함수 + 모델 준비

- **Recall@k**: 상위 k개 안에 정답 문단이 하나라도 있는 질문 비율
- **MRR@10**: 첫 정답 순위의 역수 평균 (MS MARCO 공식 지표)
- **RRF**: 여러 검색 결과를 순위만으로 합치는 방법. 점수 = Σ 1 / (60 + 순위)

In [19]:
from sentence_transformers import SentenceTransformer, CrossEncoder
import bm25s

def metrics(ranked, qs, ks=(1, 5, 10, 100)):
    res = {f'R@{k}': 0.0 for k in ks}
    res['MRR@10'] = 0.0
    for pids, q in zip(ranked, qs):
        gold = set(q['gold_pids'])
        for k in ks:
            res[f'R@{k}'] += bool(gold & set(pids[:k]))
        for r, p in enumerate(pids[:10]):
            if p in gold:
                res['MRR@10'] += 1 / (r + 1)
                break
    return {k: round(v / len(qs), 4) for k, v in res.items()}

def rrf(*rankings, k=RRF_K, top=K):
    out = []
    for lists in zip(*rankings):
        score = Counter()
        for lst in lists:
            for r, p in enumerate(lst):
                score[p] += 1 / (k + r + 1)
        out.append([p for p, _ in score.most_common(top)])
    return out

emb_model = SentenceTransformer(EMB_MODEL, device=DEVICE)
if DEVICE == 'cuda':
    emb_model.half()
emb_model.max_seq_length = 256

def encode(strings, prefix=''):
    return emb_model.encode([prefix + s for s in strings], batch_size=128, normalize_embeddings=True,
                            convert_to_numpy=True).astype('float32')

if added:
    index.add(encode(added))                     # 메모리에서만 추가 (드라이브 파일은 그대로)
assert index.ntotal == len(texts)

def dense_search(vecs, top=K):
    _, ids = index.search(vecs, top)
    return ids.tolist()

t = time.time()
bm25 = bm25s.BM25()
bm25.index(bm25s.tokenize(texts, stopwords='en', show_progress=False), show_progress=False)
print(f'BM25 인덱스 {time.time() - t:.0f}초')

def bm25_search(strings, top=K):
    ids, _ = bm25.retrieve(bm25s.tokenize(strings, stopwords='en', show_progress=False), k=top, show_progress=False)
    return ids.tolist()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

DEBUG:bm25s:Building index from IDs objects


BM25 인덱스 12초


## 6. 1단계 검색 (Q-bm25, Q-dense, Q-hybrid)

In [20]:
queries = [q['query'] for q in qs]
runs = {}
runs['Q-bm25']   = bm25_search(queries)
runs['Q-dense']  = dense_search(encode(queries, QUERY_PREFIX))
runs['Q-hybrid'] = rrf(runs['Q-dense'], runs['Q-bm25'])

orig = [i for i, q in enumerate(qs) if q['variant'] == '원본']
for m in runs:
    print(f'[원본] {m:9s}', metrics([runs[m][i] for i in orig], [qs[i] for i in orig]))

[원본] Q-bm25    {'R@1': 0.1457, 'R@5': 0.3629, 'R@10': 0.4914, 'R@100': 0.8171, 'MRR@10': 0.2385}
[원본] Q-dense   {'R@1': 0.2686, 'R@5': 0.5971, 'R@10': 0.7314, 'R@100': 0.9743, 'MRR@10': 0.4127}
[원본] Q-hybrid  {'R@1': 0.2286, 'R@5': 0.4971, 'R@10': 0.66, 'R@100': 0.9629, 'MRR@10': 0.3508}


## 7. 재정렬 (cross-encoder)

cross-encoder는 질문과 문단을 **한 번에 같이 읽고** 관련도를 매깁니다. 1단계 Q-dense 후보 100개만 다시 줄 세웁니다.
(이전 v1.1 실험에서 1단계는 dense가 가장 좋았습니다. hybrid R@10 0.650 < dense 0.754)

In [21]:
CE_BATCH = 256

def check_reranker(ce):
    """관련 있는 문장이 더 높은 점수를 받는지, 점수가 nan이 아닌지 확인합니다."""
    s = ce.predict([('how many people live in berlin', 'Berlin has a population of about 3.5 million people.'),
                    ('how many people live in berlin', 'The cat sat on the mat.')], batch_size=CE_BATCH)
    assert np.isfinite(s).all(), f'재정렬 점수가 nan입니다: {s}. 라이브러리 버전 문제일 수 있어요.'
    assert s[0] > s[1], f'관련 문장 점수가 더 낮습니다: {s}'
    print('  점검 통과:', np.round(s, 2))

def rerank(ce, qs, cand_lists):
    out = []
    for q, cands in zip(qs, cand_lists):
        scores = ce.predict([(q['query'], texts[p]) for p in cands], batch_size=CE_BATCH, show_progress_bar=False)
        assert np.isfinite(scores).all(), '재정렬 중 nan 점수 발생'
        out.append([cands[i] for i in np.argsort(-scores)])
    return out

for name in RERANKERS:
    t = time.time()
    ce = CrossEncoder(name, max_length=256, device=DEVICE)
    check_reranker(ce)
    runs[f'Q-dense → {name.split("/")[-1]}'] = rerank(ce, qs, runs['Q-dense'])
    print(f'{name} 재정렬 {time.time() - t:.0f}초')
    del ce
assert FINAL in runs, FINAL

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

... (10 outputs left out)

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/279 [00:00<?, ?B/s]

  점검 통과: [1. 0.]
BAAI/bge-reranker-base 재정렬 1923초


## 8. 결과 표

- 변형별 × 방법별 전체 지표
- 변형별 MRR@10 / R@10 한눈에 보기
- 원본 질문의 유형별 점수 (`FINAL` 방법)

In [22]:
def subset(cond):
    idx = [i for i, q in enumerate(qs) if cond(q)]
    return {m: metrics([r[i] for i in idx], [qs[i] for i in idx]) for m, r in runs.items()}, len(idx)

by_variant = {}
for v in VARIANTS:
    scores, n = subset(lambda q: q['variant'] == v)
    by_variant[v] = scores
    print(f'\n[{v}] 질문 {n}개')
    display(pd.DataFrame(scores).T)

for key in ['MRR@10', 'R@10']:
    print(f'\n{key} (행 = 방법, 열 = 변형)')
    display(pd.DataFrame({v: {m: s[key] for m, s in by_variant[v].items()} for v in VARIANTS}))

cats = Counter(q['category'] for q in qs if q['variant'] == '원본')
by_cat = {}
for c, n in cats.most_common():
    scores, _ = subset(lambda q: q['variant'] == '원본' and q['category'] == c)
    by_cat[c] = {'n': n, 'Q-dense R@10': scores['Q-dense']['R@10'],
                 'FINAL R@10': scores[FINAL]['R@10'], 'FINAL MRR@10': scores[FINAL]['MRR@10']}
print(f'\n원본 질문 유형별 (FINAL = {FINAL})')
display(pd.DataFrame(by_cat).T)


[원본] 질문 350개


,R@1,R@5,R@10,R@100,MRR@10
Q-bm25,0.1457,0.3629,0.4914,0.8171,0.2385
Q-dense,0.2686,0.5971,0.7314,0.9743,0.4127
Q-hybrid,0.2286,0.4971,0.6600,0.9629,0.3508
Q-dense → ms-marco-MiniLM-L-6-v2,0.2829,0.6457,0.8114,0.9743,0.4308
Q-dense → bge-reranker-base,0.2714,0.6143,0.7800,0.9743,0.4265



[구어체] 질문 350개


,R@1,R@5,R@10,R@100,MRR@10
Q-bm25,0.1343,0.3486,0.4543,0.7857,0.2224
Q-dense,0.2543,0.5857,0.7171,0.9686,0.4003
Q-hybrid,0.2200,0.4857,0.6457,0.9514,0.3368
Q-dense → ms-marco-MiniLM-L-6-v2,0.2571,0.6343,0.7657,0.9686,0.4105
Q-dense → bge-reranker-base,0.2571,0.5943,0.7314,0.9686,0.3975



[쉬운 말] 질문 350개


,R@1,R@5,R@10,R@100,MRR@10
Q-bm25,0.1057,0.2971,0.4143,0.7600,0.1855
Q-dense,0.2343,0.5743,0.7114,0.9714,0.3799
Q-hybrid,0.2171,0.4629,0.6257,0.9514,0.3268
Q-dense → ms-marco-MiniLM-L-6-v2,0.2600,0.6086,0.7714,0.9714,0.4115
Q-dense → bge-reranker-base,0.2286,0.5629,0.7086,0.9714,0.3685


... (4 outputs left out)


R@10 (행 = 방법, 열 = 변형)


,원본,구어체,쉬운 말,핵심 단어 회피
Q-bm25,0.4914,0.4543,0.4143,0.0086
Q-dense,0.7314,0.7171,0.7114,0.0000
Q-hybrid,0.6600,0.6457,0.6257,0.0029
Q-dense → ms-marco-MiniLM-L-6-v2,0.8114,0.7657,0.7714,0.0029
Q-dense → bge-reranker-base,0.7800,0.7314,0.7086,0.0057



원본 질문 유형별 (FINAL = Q-dense → ms-marco-MiniLM-L-6-v2)


,n,Q-dense R@10,FINAL R@10,FINAL MRR@10
정의·의미,83.0,0.8072,0.8795,0.5097
원인,47.0,0.6596,0.7660,0.3897
해부·생리,34.0,0.6765,0.7353,0.3870
검사·정상수치,31.0,0.6129,0.7742,0.4267
기타,28.0,0.5714,0.6786,0.2896
기간·시기,19.0,0.7895,0.8947,0.3969
효능·용도,17.0,0.7647,0.7647,0.4966
약물,15.0,0.7333,0.8000,0.3978
증상,14.0,0.9286,0.9286,0.3873
예/아니오 확인,14.0,0.9286,0.9286,0.6452


## 9. 저장 + 생성 단계로 넘길 파일

- `results.json`, `results_summary.md`: 점수 표
- `test_top20.jsonl`: `FINAL` 방법으로 뽑은 **질문별 상위 20개 문단 번호** (1,400개 질문 전부, 변형 정보 포함)
- `added_passages.jsonl`: DB에 없어서 메모리에 추가한 정답 문단. pid가 `len(corpus)`부터 이어지므로, 생성 단계에서 문단 원문을 찾을 때 함께 써야 합니다.

In [23]:
def to_md(df):
    try:
        return df.to_markdown()
    except ImportError:
        return '```\n' + df.to_string() + '\n```'

mrr = pd.DataFrame({v: {m: s['MRR@10'] for m, s in by_variant[v].items()} for v in VARIANTS})
r10 = pd.DataFrame({v: {m: s['R@10'] for m, s in by_variant[v].items()} for v in VARIANTS})

with open(f'{OUT_DIR}/results.json', 'w', encoding='utf-8') as f:
    json.dump({'test_path': TEST_PATH, 'db_dir': DB_DIR, 'n_passages_db': n_db, 'n_added_gold': len(added),
               'n_queries': len(qs), 'final_method': FINAL,
               'by_variant': by_variant, 'by_category_original': by_cat}, f, indent=2, ensure_ascii=False)

summary = '\n'.join([
    f"# 검색 베이스라인: holdout test 350 + 변형 ({time.strftime('%Y-%m-%d %H:%M')})", '',
    f'- DB: 문단 {n_db:,}개 (+ 없던 정답 문단 {len(added)}개 메모리 추가) / 질문 {len(qs)}개',
    f'- 생성 단계로 넘긴 방법: {FINAL}',
    '- `핵심 단어 회피`는 주제어를 지운 질문이라 검색 점수가 낮은 게 정상',
    '- MiniLM 리랭커는 MS MARCO train으로 학습됨 → 테스트 질문과 겹칠 수 있음(점수 과대 가능)', '',
    '## MRR@10', '', to_md(mrr), '', '## R@10', '', to_md(r10), '',
    '## 원본 질문 유형별', '', to_md(pd.DataFrame(by_cat).T), '',
    *sum([[f'## {v} 전체 지표', '', to_md(pd.DataFrame(by_variant[v]).T), ''] for v in VARIANTS], []),
])
with open(f'{OUT_DIR}/results_summary.md', 'w', encoding='utf-8') as f:
    f.write(summary)

save_jsonl([{'qid': q['qid'], 'group': q['group'], 'variant': q['variant'], 'query': q['query'],
             'gold_pids': q['gold_pids'], 'top20': [int(p) for p in r[:TOP_SAVE]]}
            for q, r in zip(qs, runs[FINAL])], f'{OUT_DIR}/test_top20.jsonl')
save_jsonl([{'pid': n_db + i, 'text': t} for i, t in enumerate(added)], f'{OUT_DIR}/added_passages.jsonl')
print(summary)

# 검색 베이스라인: holdout test 350 + 변형 (2026-11-02 04:16)

- DB: 문단 142,605개 (+ 없던 정답 문단 48개 메모리 추가) / 질문 1400개
- 생성 단계로 넘긴 방법: Q-dense → ms-marco-MiniLM-L-6-v2
- `핵심 단어 회피`는 주제어를 지운 질문이라 검색 점수가 낮은 게 정상
- MiniLM 리랭커는 MS MARCO train으로 학습됨 → 테스트 질문과 겹칠 수 있음(점수 과대 가능)

## MRR@10

|                                  |   원본 |   구어체 |   쉬운 말 |   핵심 단어 회피 |
|:---------------------------------|-------:|---------:|----------:|-----------------:|
| Q-bm25                           | 0.2385 |   0.2224 |    0.1855 |           0.0017 |
| Q-dense                          | 0.4127 |   0.4003 |    0.3799 |           0      |
| Q-hybrid                         | 0.3508 |   0.3368 |    0.3268 |           0.0003 |
| Q-dense → ms-marco-MiniLM-L-6-v2 | 0.4308 |   0.4105 |    0.4115 |           0.0007 |
| Q-dense → bge-reranker-base      | 0.4265 |   0.3975 |    0.3685 |           0.0019 |

## R@10

|                                  |   원본 |   구어체 |   쉬운 말 |   핵심 단어 회피 |
|:---------------------------------|-----

## 10. 예시: 같은 질문의 변형 4개, 재정렬 결과 상위 3개 (✅ = 정답 문단)

In [24]:
g = qs[0]['group']
for i, q in enumerate(qs):
    if q['group'] != g:
        continue
    print(f"[{q['variant']}] {q['query']}")
    for p in runs[FINAL][i][:3]:
        mark = '✅' if p in q['gold_pids'] else '  '
        print(f'  {mark} [{p}] {texts[p][:140]}')
    print()

[원본] what does antiviral medication do
  ✅ [27948] Antiviral drugs are a class of medication used specifically for treating viral infections. Like antibiotics for bacteria, specific antiviral
     [27945] Antiviral drugs. Antiviral drugs are medicines that cure or control virus infections. There are several different drugs in the antiviral fam
     [27943] Antiviral drugs are a type of medication used specifically for treating viral infections. They act by killing or preventing the growth of vi

[구어체] Quick question: what does antiviral medication do?
  ✅ [27948] Antiviral drugs are a class of medication used specifically for treating viral infections. Like antibiotics for bacteria, specific antiviral
     [27945] Antiviral drugs. Antiviral drugs are medicines that cure or control virus infections. There are several different drugs in the antiviral fam
     [77542] Antivirals are medications that keep viruses from reproducing inside the body. To date, relatively few effective antiviral

## 11. 드라이브 저장 마무리 + 런타임 종료

결과 파일은 앞 셀에서 드라이브에 저장됩니다. Run All로 실행했을 때 런타임이 꺼지지 않도록, 바로 아래 셀이 일부러 에러를 냅니다. 끄려면 아래 셀을 건너뛰고 그다음 셀을 직접 실행하세요.

In [25]:
raise SystemExit('여기서 멈춤: 런타임을 끄려면 다음 셀을 직접 실행하세요')

SystemExit: 여기서 멈춤: 런타임을 끄려면 다음 셀을 직접 실행하세요

/usr/local/lib/python3.13/dist-packages/IPython/core/interactiveshell.py:3561: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [ ]:
from google.colab import drive, runtime
drive.flush_and_unmount()   # 결과 파일 드라이브 저장 마무리
runtime.unassign()          # 런타임 종료